# Dropbox shared-folder downloader

Stream every file from a Dropbox shared folder, resample to mono / 16 kHz, write it under `recordings/`, and delete the raw download right after so we never hold the un-resampled copy on disk.

Auth: OAuth2 refresh-token flow. `.env` must contain `DROPBOX_APP`, `DROPBOX_SECRET`, and `DROPBOX_REFRESH_TOKEN`. If the refresh token is empty, run the "One-time auth bootstrap" cell once and paste the printed value into `.env`.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

import dropbox
import librosa
import pyrootutils
import soundfile as sf
from dotenv import load_dotenv
from dropbox.files import FileMetadata, SharedLink
from tqdm.auto import tqdm

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)
load_dotenv()

from building.utils.constants import SAMPLE_RATE

2026-05-22 10:22:35.276538: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-22 10:22:35.293786: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-22 10:22:35.317273: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-22 10:22:35.317308: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-22 10:22:35.331914: I tensorflow/core/platform/cpu_feature_gua

## Config

In [3]:
SHARED_URL = "https://www.dropbox.com/scl/fo/5gk3iczix2jmlzoonvcwe/h/data/robert_lachlan/R21?e=2&subfolder_nav_tracking=1&dl=0"

OUT_DIR = ROOT / "recordings"
TMP_DIR = ROOT / "recordings" / "_tmp"
OUT_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_BYTES = 4 * 1024 * 1024
AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg", ".aif", ".aiff", ".m4a"}

print(f"target sample rate: {SAMPLE_RATE} Hz, mono")
print(f"output: {OUT_DIR}")

target sample rate: 16000 Hz, mono
output: /home/nathan/Documents/multi-chirp/recordings


## One-time auth bootstrap

Run this once if `DROPBOX_REFRESH_TOKEN` in `.env` is empty. It prints an authorize URL — open it, click "Allow", paste the returned code back, and it will exchange the code for a refresh token. Copy that refresh token into `.env` as `DROPBOX_REFRESH_TOKEN=...`, then skip this cell on future runs.

In [4]:
import requests
from urllib.parse import urlencode

APP_KEY = os.environ["DROPBOX_APP"]
APP_SECRET = os.environ["DROPBOX_SECRET"]

auth_url = "https://www.dropbox.com/oauth2/authorize?" + urlencode({
    "client_id": APP_KEY,
    "response_type": "code",
    "token_access_type": "offline",
})
print("1. Open this URL, click Allow, copy the code shown:")
print(f"   {auth_url}")
code = input("2. Paste the code here: ").strip()

resp = requests.post(
    "https://api.dropbox.com/oauth2/token",
    data={"code": code, "grant_type": "authorization_code"},
    auth=(APP_KEY, APP_SECRET),
    timeout=30,
)
resp.raise_for_status()
refresh_token = resp.json()["refresh_token"]
print()
print("3. Paste this line into .env (replace the empty DROPBOX_REFRESH_TOKEN):")
print(f"DROPBOX_REFRESH_TOKEN={refresh_token}")

1. Open this URL, click Allow, copy the code shown:
   https://www.dropbox.com/oauth2/authorize?client_id=5c45je73j338z0w&response_type=code&token_access_type=offline


HTTPError: 400 Client Error: Bad Request for url: https://api.dropbox.com/oauth2/token

## Connect

Uses the refresh token in `.env` — the SDK transparently mints a fresh short-lived access token whenever it expires, so this cell keeps working indefinitely.

In [5]:
dbx = dropbox.Dropbox(
    oauth2_refresh_token=os.environ["DROPBOX_REFRESH_TOKEN"],
    app_key=os.environ["DROPBOX_APP"],
    app_secret=os.environ["DROPBOX_SECRET"],
)
dbx.users_get_current_account().email

'nathan.duboisset@gmail.com'

## Parse the shared URL

A Dropbox `scl` shared URL like `https://www.dropbox.com/scl/fo/<id>/h/<subpath>?...` is **a single link to the deepest folder** (here, R21) — the `/h/<subpath>` portion isn't a separate path argument, it's part of the link identifier. So we just strip the query string and keep the rest.</cell id="cell-9">

In [6]:
def shared_link_url(url: str) -> str:
    parts = urlsplit(url)
    return urlunsplit((parts.scheme, parts.netloc, parts.path, "", ""))


LINK_URL = shared_link_url(SHARED_URL)
print("link url:", LINK_URL)

link url: https://www.dropbox.com/scl/fo/5gk3iczix2jmlzoonvcwe/h/data/robert_lachlan/R21


## List files

In [7]:
from dropbox.files import FolderMetadata


def list_shared_folder(
    dbx: dropbox.Dropbox, link_url: str
) -> list[tuple[str, FileMetadata]]:
    """Return (link_relative_path, FileMetadata) for every file under the link.

    Walks subdirectories manually — Dropbox doesn't allow recursive=True with
    shared links, and shared-link entries have `path_lower=None`, so we build
    link-relative paths ourselves from parent + name.
    """
    sl = SharedLink(url=link_url)
    files: list[tuple[str, FileMetadata]] = []
    stack = [""]
    while stack:
        path = stack.pop()
        res = dbx.files_list_folder(path=path, shared_link=sl)
        while True:
            for e in res.entries:
                child = f"{path}/{e.name}"
                if isinstance(e, FileMetadata):
                    files.append((child, e))
                elif isinstance(e, FolderMetadata):
                    stack.append(child)
            if not res.has_more:
                break
            res = dbx.files_list_folder_continue(res.cursor)
    return files


all_files = list_shared_folder(dbx, LINK_URL)
audio_files = [
    (p, f) for (p, f) in all_files if Path(f.name).suffix.lower() in AUDIO_EXTS
]
print(f"{len(all_files)} total entries, {len(audio_files)} audio files")
for p, f in audio_files[:5]:
    print(f"  {p}  ({f.size/1e6:.1f} MB)")

98 total entries, 98 audio files
  /R21_2022_02_24_08_03_24.wav  (317.5 MB)
  /R21_2022_02_22_08_46_50.wav  (317.5 MB)
  /R21_2022_02_25_07_00_01.wav  (317.5 MB)
  /R21_2022_02_16_08_52_05.wav  (317.5 MB)
  /R21_2022_02_22_07_55_45.wav  (317.5 MB)


## Stream + resample

For each file:
1. Stream-download to `recordings/_tmp/` (chunked, never read whole file into memory at once).
2. Load + resample to mono / `SAMPLE_RATE` with librosa (streamed off disk, not the whole raw bytes again).
3. Write the resampled file under `recordings/...`.
4. Delete the raw temp file immediately.

In [8]:
import shutil


def stream_download(
    dbx: dropbox.Dropbox, link_url: str, file_path: str, dest: Path
) -> None:
    _md, resp = dbx.sharing_get_shared_link_file(url=link_url, path=file_path)
    try:
        with open(dest, "wb") as out:
            for chunk in resp.iter_content(chunk_size=CHUNK_BYTES):
                if chunk:
                    out.write(chunk)
    finally:
        resp.close()


def process(link_path: str, file_md: FileMetadata) -> str:
    # Flatten: drop any subfolder structure from the shared link and write
    # straight into OUT_DIR, so later `glob` calls don't have to recurse.
    out_path = (OUT_DIR / file_md.name).with_suffix(".wav")
    # Skip only if the resampled output exists AND is non-empty — a 0-byte
    # file from a previously crashed run should be re-processed, not skipped.
    if out_path.exists() and out_path.stat().st_size > 0:
        return "skip"

    tmp = TMP_DIR / file_md.id.replace(":", "_") / file_md.name
    tmp.parent.mkdir(parents=True, exist_ok=True)
    try:
        # Reuse a tmp from a previous interrupted run only if its size
        # matches the Dropbox metadata; otherwise re-download.
        if not (tmp.exists() and tmp.stat().st_size == file_md.size):
            if tmp.exists():
                tmp.unlink()
            stream_download(dbx, LINK_URL, link_path, tmp)
        sig, _ = librosa.load(
            str(tmp), sr=SAMPLE_RATE, mono=True, res_type="kaiser_fast"
        )
        sf.write(str(out_path), sig, SAMPLE_RATE, subtype="PCM_16")
    finally:
        if tmp.exists():
            tmp.unlink()
        try:
            tmp.parent.rmdir()
        except OSError:
            pass
    return "ok"


stats = {"ok": 0, "skip": 0, "err": 0}
for link_path, f in tqdm(audio_files, desc="R21"):
    try:
        stats[process(link_path, f)] += 1
    except Exception as e:
        stats["err"] += 1
        print(f"  fail {link_path}: {type(e).__name__}: {e}")

print(stats)
shutil.rmtree(TMP_DIR, ignore_errors=True)

R21:   0%|          | 0/98 [00:00<?, ?it/s]

/tmp/ipykernel_50927/908824608.py:35: UserWarning: PySoundFile failed. Trying audioread instead.
  sig, _ = librosa.load(
/home/nathan/Documents/multi-chirp/.venv/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


  fail /R21_2022_02_16_07_53_17.wav: EOFError: 
{'ok': 27, 'skip': 70, 'err': 1}
